## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://i.imgur.com/Q8HEZn0.png)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

---

# 🤝 Breakout Room #1
## Deep Research Foundations

In this breakout room, we'll understand the architecture and components of the Open Deep Research system.

## Task 1: Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value
    
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

LangSmith tracing enabled. Project: AIE9 - Deep Agents - d5fab766
OpenAI API key set


## Task 2: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

## Task 3: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

## Task 4: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 5: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## ❓ Question #1:

Explain the interrelationships between the three states (Agent, Supervisor, Researcher). Why don't we just make a single huge state?

- The state structure follows a hierarchical subgraph pattern.
- AgentState (Main graph) --> SupervisorState (Subgraph)  --> ResearcherState (can be multiple subgraphs running in parallel)


|Aspect| AgentState | SupervisorState | ResearcherState |
|---------|------------|-----------------|-----------------|
|Purpose|Main orchestrator and the top level state for the entire workflow|Manages research delegation and coordination|Focussed research on specific topics|
|Contains|Messages and research data - supervisor messages, research brief, raw notes, notes, final report|Research iterations, supervisor messages, research brief, raw notes, notes|Researcher messages, tool call iterations, research topic, compressed research, raw notes|
|Used By|Main workflow nodes - clarify_with_user, write_research_brief, final_report_generation|Supervisor subgraph that decides what to research and delegates tasks|Researcher subgraph that performs actual research work|

Interrelationship between the 3 states:
1. AgentState --> SupervisorState: When write_research_brief completes, it transitions to the supervisor subgraph, which extracts relevant fields from AgentState to initialize SupervisorState.

2. SupervisorState --> ResearcherState: When the supervisor calls ConductResearch, it spawns researcher subgraphs in parallel, each with its own ResearcherState initialized with a specific research_topic.

3. ResearcherState --> SupervisorState: Researchers return compressed_research and raw_notes, which get aggregated back into SupervisorState.

4. SupervisorState → AgentState: When research completes, the supervisor subgraph returns results that update AgentState with notes and research_brief.

Using a single large monolithic state is hard to manage inprodcution, making scaling, debugging and deployment significantly more complex. Separate states keep each role focused on only what it needs (separation of concerns), allows multiple researchers to run independently in parallel and makes updates, failures easier to isolate. This design is clear, safer and maintainable.


## ❓ Question #2:

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

Advantages: Reusable code in multiple notebooks, easier to maintain, mirrors real-world development practices, library code can be unit tested independently.

Disadavantages: Can be overwhelming for beginners and non-coders to go over multiple files to understand E2E working, can add debugging complexity if proper versions of dependencies are not installed, compilation issues etc


## 🏗️ Activity #1: Explore the Prompts

Open `open_deep_library/prompts.py` and examine one of the prompt templates in detail.

**Requirements:**
1. Choose one prompt template (clarify, brief, supervisor, researcher, compression, or final report)
2. Explain what the prompt is designed to accomplish
3. Identify 2-3 key techniques used in the prompt (e.g., structured output, role definition, examples)
4. Suggest one improvement you might make to the prompt

**YOUR CODE HERE** -

**lead_researcher_prompt** : Why I choose lead_researcher_prompt? I want to understand how the lead supervisor agent handles delegation, parallel sub-agent execution and the use of ReAct pattern within the prompt.

What does the prompt do?
 - Analyzes the user question and breaks it down.
 - Delegates focused tasks to sub-agents via ConductResearch.
 - Uses think_tool before and after calling ConductResearch for planning and reflection.
 - Decides when research is complete and calls ResearchComplete
 - Either calls single sub-agent or parallel sub-agents for execution based on the user research question.
 - Manages resource limits via max_researcher_iterations, max_concurrent_research_units
 
Key techniques used:
1. Role definition and context setting - Clearly says "You are a research supervisor" and sets clear expectations.
2. Structured task decomposition: uses xml styles tags "<Task></Task><Available Tools> <Instructions>" etc to organize the instructions. This separates concerns.
3. ReACT Pattern: Uses think_tool before and after ConductResearch calls. This enforces planning and reflection and improves decision quality.
4. Clearly defines the hard limits and budget constraints

Suggested Improvement:
- Think we should add explicit success criteria for when to call ResearchComplete as "When you are completely satisfied" is subjective and can lead to over-research or premature stopping.

"""
<Success Criteria>
Call ResearchComplete only when all of the following are true:
- Every part of user's question is answered
- Each major claim is supported by at least one reliable source
- No unresolved gaps remain after evaluation
- You can confidently answer the question without further delegation

Do NOT continue researching for marginal improvements once these criteria are met.
</Success Criteria>"""


---

# 🤝 Breakout Room #2
## Building & Running the Researcher

In this breakout room, we'll explore the node functions, build the graph, and run wellness research.

## Task 6: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [6]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [7]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [11]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [12]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [13]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 7: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [14]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 8: Running the Deep Researcher

Now let's see the system in action! We'll use it to research wellness strategies for improving sleep quality.

### Setup

We need to:
1. Set up the wellness research request
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [15]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (1 concurrent researcher for cost control)
- **Clarification enabled** (will ask if research scope is unclear)

In [16]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "gpt-4.1",
        "research_model_max_tokens": 4000,
        
        "compression_model": "gpt-4.1",
        "compression_model_max_tokens": 3000,
        
        "final_report_model": "gpt-4.1",
        "final_report_model_max_tokens": 3000,
        
        "summarization_model": "gpt-4.1-mini",
        "summarization_model_max_tokens": 3000,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researcher
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: OpenAI")
print(f"  - Max Concurrent Researchers: 1")
print(f"  - Max Iterations: 2")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: OPenAI
  - Max Concurrent Researchers: 1
  - Max Iterations: 2
  - Search API: Tavily


### Execute the Wellness Research

Now let's run the research! We'll ask the system to research evidence-based strategies for improving sleep quality.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [17]:
# Create our wellness research request
research_request = """
I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please research the best evidence-based strategies for improving sleep quality and create a comprehensive sleep improvement plan for me.
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

Thank you for providing the details about your current sleep habits and your goal to improve sleep quality. You mentioned inconsistent bedtime, phone use in bed, and feeling tired in the mornings. I have enough information to proceed and will now start researching evidence-based strategies to create a comprehensive sleep improvement plan tailored to your situation.

Node: write_research_brief

Research Brief Generated:
I want to improve my sleep quality. My current habits include going to bed at inconsistent times (between 10pm and 1am), using my phone in bed before sleep, and often feeling tired in the morning. Please research the best evidence-based strategies for improving sleep quality, with particular focus on interventions addressing inconsistent bedtimes, pre-sleep phone use, and morning tiredness. Create a comprehensive sleep improvement plan tailored to these specific challenges. If additional persona...

Node: research_

# Comprehensive Sleep Improvement Plan: Evidence-Based Strategies for Inconsistent Bedtimes, Pre-Sleep Phone Use, and Morning Tiredness

## Overview

Improving sleep quality is a multi-faceted goal, especially when faced with challenges like inconsistent bedtimes, phone use in bed, and persistent morning tiredness. These habits interact with the body’s circadian regulation, sleep architecture, and daytime functioning. The following plan synthesizes high-quality academic and health authority research to address these barriers, supplying both general and targeted interventions. Where relevant, the need for further personalization is highlighted.

---

## 1. Importance of Consistent Bedtimes

Inconsistent bedtimes disrupt the circadian rhythms that regulate sleep timing and quality. Research indicates:

- Greater variability in bedtimes is significantly associated with decreased average sleep time and poorer sleep quality, even independently of time spent asleep[1].
- Consistent bedtime and wake-up times are linked to better alertness, lower risk of mood disorders, improved physical health, and cognitive performance[2].
- Healthy adults should target 7–9 hours of sleep nightly, maintaining consistency even on weekends. “Catch-up” weekend sleep cannot fully reverse chronic sleep debt[3][4].

**Recommendations:**

- Set and maintain a regular bedtime and wake-up time, allowing for 7–9 hours of sleep, even on non-workdays or weekends[2].
- Use alarms or reminders to establish routine, including prompts to begin a wind-down process at the same time nightly.
- Adjust gradually: shift bedtime by 15–30 minutes earlier every few days if needed until the ideal schedule is reached.

---

## 2. Reducing or Eliminating Phone Use in Bed

Nighttime phone use introduces two major issues: blue light exposure suppresses melatonin, delaying sleep onset and reducing sleep quality; interactive/engaging content heightens alertness and stress[5][6].

**Impacts of Phone Use:**

- Phone screens emit blue light that delays melatonin production and disturbs circadian rhythms, leading to shortened REM sleep and next-day fatigue[5][7].
- Nighttime phone usage is linked to later sleep onset, shorter sleep duration, poorer sleep quality, and increased daytime tiredness[8].
- The effect is strongest with interactive devices (smartphones, tablets) compared to passive screens (e.g., TV)[7].

**Recommendations:**

- Establish a “digital curfew”: stop using all screens (including phones, tablets, and computers) at least 30–60 minutes before bed[6][9].
- Remove your phone from the bedroom, or at minimum, from the bedside area. Use traditional alarm clocks if needed.
- Activate “Do Not Disturb” modes in the evening to reduce temptation and limit disturbances.
- Replace screen time with relaxing pre-bed routines: reading (on paper, not backlit screens), gentle stretches, or meditation practices[10].
- If phone use is necessary, enable night-shift/blue-light filter modes and dim screens, but strive to minimize any interaction[6].

---

## 3. Alleviating Morning Tiredness and Improving Alertness

Morning tiredness (even after adequate sleep) may have several causes, including poor sleep quality, sleep inertia, inconsistent sleep schedules, and suboptimal morning habits[11].

**Evidence-Based Interventions:**

- **Morning Light Exposure:** Natural daylight in the morning resets circadian rhythm and increases serotonin, enhancing alertness and mood[12].
- **Hydration:** Drink water soon after waking; overnight dehydration can cause sluggishness[13].
- **Balanced Breakfast:** Prefer high-fiber, protein-rich, low-sugar foods (e.g., eggs, oatmeal, nuts, whole grains, fruit) to stabilize energy[14].
- **Physical Activity:** Morning exercise (even light stretching or walking) boosts energy and can promote better sleep at night[15].
- **Optimize Wake-Up Routine:** Avoid repeatedly hitting “snooze”—this fragments late-stage sleep and worsens inertia[13].
- **Prior Evening Sleep Hygiene:** Adhering to regular sleep and limiting evening stimulants (caffeine, alcohol, heavy meals) also improves morning waking[3][4].

If severe fatigue persists despite optimal sleep practices, underlying medical issues (e.g., sleep apnea, depression, thyroid dysfunction) should be ruled out[11][16].

---

## 4. General Sleep Hygiene and Environmental Factors

High-quality sleep is also shaped by the sleep environment and other modifiable behaviors[17]:

- **Bedroom Environment:** Ensure your room is dark, quiet, and cool. Block external light sources and remove electronics[18].
- **Limited Napping:** Avoid late afternoon naps, which can delay sleep onset at night; if napping, keep it brief (<30 minutes, before 3 p.m.)[19].
- **Stimulus Control:** Use the bed only for sleep (and sex), getting out of bed if unable to fall asleep after ~20 minutes[17].
- **Relaxation Techniques:** Deep breathing, progressive muscle relaxation, or mindfulness before bed can help separate wakefulness from sleep time[10].

---

## 5. Special Considerations: The Role of Personal Factors

Certain unmentioned factors can significantly impact sleep and fatigue; individualization may be required in the following cases:

- **Medical Conditions:** Conditions such as chronic insomnia, obstructive sleep apnea, or depression require tailored evaluation and management[16].
- **Age:** Although sleep duration needs remain relatively stable in adulthood, circadian rhythms may change with age, sometimes requiring schedule adaptation[16].
- **Work Schedules:** Shift work or inconsistent daytime responsibilities may require the use of light exposure strategies and environmental controls (e.g., blackout curtains, earplugs)[18].
- **Chronotype:** Personal biological preference for morning or evening may influence ideal timing, but overall regularity remains crucial[20].

Consultation with a healthcare professional is recommended if persistent sleep disruption or significant daytime impairment continues despite these interventions.

---

## 6. Step-by-Step Sleep Improvement Action Plan

1. **Establish a fixed bedtime/wake-up routine** (targeting 7–9 hours, within 30 minutes across days).  
2. **Implement a digital curfew**: cease phone and device use at least 60 minutes before bed, remove from bedroom if possible.  
3. **Create a calming pre-sleep routine**: read a book, journal, or use relaxation/breathing exercises.  
4. **Optimize bedroom environment**: ensure it is dark, cool, and quiet; remove light and electronic distractions.  
5. **Eat a healthy, light dinner** and avoid caffeine/alcohol within 4–6 hours of bedtime.  
6. **Upon waking**, expose yourself to natural light, hydrate, and eat a balanced breakfast. Add some light movement to boost alertness.  
7. **Exercise regularly** (ideally early in the day); avoid intense evening workouts if you’re prone to insomnia.  
8. **Limit daytime naps** and avoid long or late naps.  
9. **If you can’t sleep, get up** and do something relaxing until drowsy; don’t stay awake in bed.  
10. **Monitor progress**, and seek medical guidance if sleep/wake problems persist despite sustained effort.

---

## 7. When to Seek Medical Advice

- If, after adhering to these evidence-based strategies for several weeks, significant difficulties with sleep or morning alertness remain, consult a healthcare provider.
- Warning signs needing evaluation include loud snoring, breathing pauses during sleep, chronic insomnia, severe depression/anxiety, or extreme fatigue.

---

## Sources

[1] Effects of an irregular bedtime schedule on sleep quality, daytime sleepiness, and fatigue among university students: https://pmc.ncbi.nlm.nih.gov/articles/PMC2718885/  
[2] Consistent Sleep Schedules with New Consensus Guideline: https://www.thensf.org/sleep-schedules-sleep-timing-guideline/  
[3] Sleep Better: Evidence-Based Strategies to Improve Sleep: https://www.linkedin.com/pulse/sleep-better-evidence-based-strategies-improve-dr-tessa-browne-bcjve  
[4] Good Sleep for Good Health | NIH News in Health: https://newsinhealth.nih.gov/2021/04/good-sleep-good-health  
[5] Using Your Phone in Bed: 3 Reasons To Avoid It: https://www.health.com/mind-body/3-reasons-not-to-sleep-with-your-phone-in-your-bed  
[6] Bedtime screen time may reduce sleep quality - Harvard Health: https://www.health.harvard.edu/staying-healthy/bedtime-screen-time-may-reduce-sleep-quality  
[7] A study on the effect of mobile phone use on sleep: https://pmc.ncbi.nlm.nih.gov/articles/PMC9707689/  
[8] Digital media use and sleep in late adolescence and young adulthood: https://www.sciencedirect.com/science/article/pii/S1087079222001551  
[9] Technology Impacts on Sleep Quality | Blog - Sleep Health Solutions: https://www.sleephealthsolutionsohio.com/blog/how-technology-use-decreases-sleep-quality/  
[10] Strategies to promote better sleep in these uncertain times - Harvard Health: https://www.health.harvard.edu/blog/strategies-to-promote-better-sleep-in-these-uncertain-times-2020032719333  
[11] Waking up tired: Causes, symptoms, and treatments: https://www.medicalnewstoday.com/articles/waking-up-tired  
[12] Shine light on sleep: Morning bright light improves nocturnal sleep: https://onlinelibrary.wiley.com/doi/abs/10.1111/jsr.13724  
[13] 13 Quick Ways to Banish Morning Fatigue: https://www.healthline.com/health/morning-fatigue-remedies  
[14] Tired in the Morning? Try These Three Things: https://greatergood.berkeley.edu/article/item/tired_in_the_morning_try_these_three_things  
[15] (PDF) Morning exercise improves sleep quality in university students: https://www.researchgate.net/publication/363099248_Morning_exercise_improves_sleep_quality_in_university_students  
[16] Does Daytime Tiredness Mean You Need More Sleep? - Sleep Foundation: https://www.sleepfoundation.org/how-sleep-works/does-daytime-tiredness-mean-you-need-more-sleep  
[17] Behavioral Strategies, Including Exercise, for Addressing Insomnia: https://pmc.ncbi.nlm.nih.gov/articles/PMC6715137/  
[18] Create a Good Sleep Environment - CDC: https://www.cdc.gov/niosh/work-hour-training-for-nurses/longhours/mod6/02.html  
[19] Sleep and circadian hygiene practices association with sleep quality: https://www.sciencedirect.com/science/article/pii/S2590142723000289  
[20] Scientists discover secret to waking up alert and refreshed: https://news.berkeley.edu/2022/11/29/scientists-discover-secret-to-waking-up-alert-and-refreshed/


Research workflow completed!


## Task 9: Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided specific details about your sleep issues, it likely proceeded without asking clarifying questions.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` to delegate to researchers
- Each delegation specified a focused research topic (e.g., sleep hygiene, circadian rhythm, blue light effects)

### Phase 4: Parallel Research
Researchers worked on their assigned topics:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive sleep improvement plan with:
- Well-structured sections
- Evidence-based recommendations
- Practical action items
- Sources for further reading

## Task 10: Key Takeaways & Next Steps

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

## ❓ Question #3:

What are the trade-offs of using parallel researchers vs. sequential research? When might you choose one approach over the other?

##### Answer:

|Aspect| Parallel Research | Sequential Research | When to choose |
|---------|------------|-----------------|-----------------|
|Main Idea|Many researchers work at same time|One or few researchers work step by step|Paraller for speed and sequential for clarity & depth|
|Speed|Faster overall|Slower|Choose paraller when speed matters|
|Depth/Clarity|can be shallow|Usually deeper and consistent|Choose sequential for careful reasoning|
|Cost & Resources|More expensive|Cheaper|Choose sequential with limited resources|
|Work duplication|Higher chance of duplicated work|Lower chance|Choose parallel only if overlap is acceptable and design prompt to reduce overlap|
|Aggregating results|Extra work to merge results|Less comparetively|Choose sequential if you want less management|

Use parallel research for speed and explore many ideas and use sequential research for careful, consistent, high quality results.


## ❓ Question #4:

How would you adapt this deep research architecture for a production wellness application? What additional components would you need?

##### Answer:
- To adapt this deep research architecture for a wellness production application, we will need to update the prompts and schemas specialized to welllness.

- We can add HealthWellnessGuide as a RAG store with existing web search for more grounded answers for wellness. We can structure the output as per wellness categories goals, habits, cautions.
- Add user authentication and authorization.
- Caching for common topic to reduce latency.
- Monitoring, logging.
- Guardrail to validate the output.
- Add required disclaimers.


## 🏗️ Activity #2: Custom Wellness Research

Using what you've learned, run a custom wellness research task.

**Requirements:**
1. Create a wellness-related research question (exercise, nutrition, stress, etc.)
2. Modify the configuration for your use case
3. Run the research and analyze the output
4. Document what worked well and what could be improved

**Experiment ideas:**
- Research exercise routines for specific conditions (bad knee, lower back pain)
- Compare different stress management techniques
- Investigate nutrition strategies for specific goals
- Explore meditation and mindfulness research

**YOUR CODE HERE**

In [18]:
# YOUR CODE HERE
# Create your own wellness research request and run it

my_wellness_request = """
I have a sedentary desk job and experience mild lower-back stiffness and neck tension,
especially at the end of the workday. I sit for long periods, often forget to stretch,
and usually do light walking 2–3 times per week.

Please research the best evidence-based strategies to:
- Reduce lower-back stiffness and neck tension for desk workers
- Design a safe, beginner-friendly mobility and strength routine I can do at home
- Suggest ergonomics and break patterns I can apply during the workday

Include clear, actionable recommendations, approximate weekly plan,
and any important safety considerations or red flags that should prompt consulting a doctor
or physical therapist.
"""

# Optionally modify the config
my_config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "gpt-4.1",
        "research_model_max_tokens": 6000,
        
        "compression_model": "gpt-4.1",
        "compression_model_max_tokens": 4500,
        
        "final_report_model": "gpt-4.1",
        "final_report_model_max_tokens": 6000,
        
        "summarization_model": "gpt-4.1-mini",
        "summarization_model_max_tokens": 6000,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researcher
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

# Run your research
# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": my_wellness_request}]},
        my_config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

Thank you for providing detailed information about your situation. You have a sedentary desk job, experience mild lower-back and neck stiffness, walk lightly a few times per week, and are seeking evidence-based strategies to reduce discomfort, a beginner-friendly at-home routine, ergonomic advice, and safety guidance. I have sufficient information to proceed with research and will now begin developing a comprehensive, actionable report for you.

Node: write_research_brief

Research Brief Generated:
I have a sedentary desk job and experience mild lower-back stiffness and neck tension, primarily at the end of my workday. I often sit for long periods, tend to forget to stretch, and currently do light walking 2–3 times per week as my main physical activity. Please research and summarize the best, evidence-based strategies to: (1) reduce lower-back stiffness and neck tension specifically for desk workers, (2) design a safe, beginner-f

# Evidence-Based Strategies for Reducing Lower-Back Stiffness and Neck Tension in Desk Workers: A Comprehensive Guide

## Overview

Sedentary desk work contributes to mild lower-back stiffness and neck tension through prolonged sitting, suboptimal posture, limited movement, and poorly adjusted workstations. Evidence indicates that a combination of ergonomic adjustments, regular movement breaks, targeted mobility and strength exercises, and safe workday routines can significantly reduce discomfort and prevent further issues. This comprehensive guide summarizes the best current research and provides clear, actionable steps for relief and prevention, including an at-home routine, ergonomic set-up, recommended break patterns, and safety advice.

---

## 1. Evidence-Based Strategies to Reduce Lower-Back Stiffness and Neck Tension

### Ergonomic Workspace Adjustments

- **Chair**: Use an adjustable chair with lumbar support to maintain the natural curve of the spine. Sit with your back fully supported, feet flat on the floor, and knees at a 90-degree angle.
- **Monitor**: Position your monitor at eye level, directly in front (centered with your nose) at arm’s length. This reduces the need to tilt your head or hunch forward.
- **Desk Height**: Adjust the desk, keyboard, and mouse so that elbows rest at 90 degrees and wrists remain straight. Keep frequently used items close to avoid overreaching.
- **Standing Desks**: If possible, alternate between sitting and standing using a sit-stand desk. Use an anti-fatigue mat if standing for longer intervals[1][2][3][4].

### Movement and Stretching

- **Microbreaks**: Take brief posture and movement breaks every 20–30 minutes, standing or stretching for 1–2 minutes.
- **Mobility Exercises**: Incorporate simple stretches specific to common deskworker problem areas:
  - Neck: Chin tucks, gentle neck rotations (look left/right), side bends, levator scapula stretch (“smell your armpit”), shoulder rolls.
  - Thoracic spine: Seated or standing twists, thoracic extensions, cat-cow stretches.
  - Hips/glutes: Hip flexor stretch, seated figure-4 stretch, glute clenches.
  - Chest: Standing or doorway pec stretch, wall angels[1][2][4][5][6][7].
- **Walking**: Frequent, short walks during the day interrupt prolonged sitting and help circulation and spine health[5].

### Strengthening and General Fitness

- **Neck and Shoulder Strengthening**: Evidence supports exercises such as resistance band rows, overhead presses, and scapular retractions to reduce neck pain.
- **Core and Glute Strength**: Regularly perform beginner bodyweight exercises such as squats, glute bridges, planks, and gentle back extensions to support the spine[3][4][5].
- **Progressive Overload**: Gradually increase intensity as tolerated. Begin with bodyweight exercises, aiming for proper form and pain-free movement before adding resistance[4][7].

### Stress Management

- Include relaxation techniques (deep breathing, mindfulness, brief meditation) to reduce muscular tension linked to stress[1].

---

## 2. Safe, Beginner-Friendly At-Home Mobility and Strength Routine

### General Principles

- **Start Small**: Consistency is more important than duration or intensity; prioritize form over reps.
- **Daily Mobility, 2–3x/Week Strength**: Evidence supports a brief daily mobility routine (5–10 minutes) and strength training twice or thrice weekly (8–12 minutes per session)[7][8].
- **Accessible Exercises**: All exercises listed require minimal or no equipment.

### Step-by-Step Sample Routine

#### Daily Mobility (5–10 minutes):

1. **Neck Circles or Rotations**: Gently rotate your head in circles or turn slowly left and right, 5 reps each direction.
2. **Chin Tucks**: Pull chin gently straight back (double chin), hold for 3 seconds, repeat 10 times.
3. **Shoulder Rolls**: 10 repetitions forward and backward each.
4. **Seated or Standing Thoracic Twist**: Cross arms in front, rotate upper body gently to each side, 5 reps per side.
5. **Cat-Cow (Seated or Hands-and-Knees)**: Arch and round the spine, 10 times.
6. **Standing Pec Stretch or Wall Angels**: Hold each stretch for 20–30 seconds.
7. **Hip Flexor Stretch**: Hold 20 seconds per side.
8. **Glute Squeeze**: Sitting or standing, contract and hold glutes for 5 seconds, repeat 8–10 times.

Take these as movement “snacks” throughout the day or as a single 10-minute sequence[6][7][8].

#### Beginner Strength Circuit (2–3x/week, 8–12 minutes):

- **Bodyweight Squats**: 8–12 reps. Stand with feet hip-width, bend knees and hips to lower down, keep back straight.
- **Wall or Desk Push-Ups**: 8–12 reps. Place hands on wall or edge of desk, perform push-ups keeping body straight.
- **Glute Bridges**: 10–15 reps. Lie on back, bend knees, lift hips while squeezing glutes, then lower.
- **Seated or Standing Rows (with resistance band or towel)**: 8–12 reps.
- **Plank (knees or regular)**: Hold for 20–30 seconds, repeat 2–3 times.
- **Bird Dog (optional, for core stability)**: Hands and knees, extend opposite arm and leg, 6–8 reps per side.

**How to Progress**: Start with 1 set of each exercise. Gradually increase to 2–3 sets as you feel comfortable. As strength improves, add repetitions, sets, or (for rows/presses) light resistance bands[4][7][9].

### Weekly Plan Example

| Day       | Morning                    | During Work (Microbreaks)                 | Evening (Strength) |
|-----------|----------------------------|-------------------------------------------|----------------------|
| Mon       | 5-min full-body mobility   | 1-2 exercises every 30–60 min             | Strength circuit     |
| Tue       | Neck, chest, back stretches| Desk exercises (shoulder/neck/hip/leg)    | Walk                |
| Wed       | 10-min mobility routine    | As above                                  | Strength circuit     |
| Thu       | Short morning stretches    | Desk mobility movements                   | Walk                |
| Fri       | 5-min full-body mobility   | As above                                  | Strength circuit     |
| Sat       | Walk and mobility          | Optional: home chores as activity         | Rest or activity     |
| Sun       | Rest or gentle mobility    |                                           | Rest                |

---

## 3. Practical Ergonomic Adjustments and Workday Break Patterns

### Workstation Setup

- **Chair**: Adjustable, lumbar support, hips high as or slightly above knees.
- **Desk**: Elbows at 90 degrees, wrists neutral, mouse/keyboard within easy reach.
- **Monitor**: Top at/below eye level, center aligned with nose, about arm’s length away.
- **Standing Option**: Alternate between sitting/standing if possible, but always avoid long, static postures. Use supportive footwear; consider an anti-fatigue mat[1][2][16][17][18].

### Movement and Break Patterns

- **Microbreaks**: Stand, stretch, or walk for 1–2 minutes every 20–30 minutes[2][17][18].
- **Longer Breaks**: At least every hour, move away from your workstation—stand up, perform gentle stretches, take a short walk[5][24].
- **Screen Breaks**: To reduce eye strain, follow the “20-20-20” rule: every 20 minutes, look at something 20 feet away for 20 seconds[16][17].
- **Standing Work Ratio**: Consider the “20-8-2” rule—20 minutes sitting, 8 minutes standing, 2 minutes moving, on rotation[17].
- **Reminders and Apps**: Use alarms, apps, sticky notes, or computer timers for break and posture reminders[10][13].

### Organizational Support

Where possible, request employer support for ergonomic equipment and multi-component approaches (ergonomic hardware, education, movement programs, and encouragement of a mobile work culture)[30].

---

## 4. Important Safety Considerations and Red Flags

- **Stop Immediately** if any exercise/stretches produce sharp, shooting, severe, or radiating pain.
- **Consult a Doctor or Physical Therapist** if you experience:
  - Persistent or worsening neck/lower back pain that does not improve within a few weeks.
  - Numbness, tingling, or weakness in your arms, legs, or groin.
  - Radiating pain down the arms or legs.
  - Problems with bladder or bowel control.
  - Unexplained fever, weight loss, or history of trauma[1][21][24][25].
- **Modify or Skip** any movement that does not feel comfortable. Never force stretches.
- **Existing Conditions**: If you have significant pre-existing musculoskeletal problems, injuries, or neurological symptoms, consult a healthcare professional before beginning any new exercise program[10][13].

---

## 5. Limitations and Research Gaps

- The certainty regarding optimal frequency/duration/type of workday breaks for preventing musculoskeletal issues is low due to inconsistent study results and variable intervention designs[28][29].
- While sit-stand desks reduce time spent sitting and have small to moderate effects on discomfort, firm conclusions about their ability to prevent chronic pain in desk workers are lacking[5][30].
- Comprehensive, multi-factor approaches show the most promise, but more high-quality long-term studies are required, especially in remote or hybrid work settings[30].

---

## 6. Summary

Reducing lower-back stiffness and neck tension for desk workers requires an integrated approach: optimize workspace ergonomics, take frequent movement breaks, engage in targeted daily mobility and regular strengthening routines, and consistently monitor symptoms. Use the detailed at-home and workday plan above, making adjustments for your unique needs, and remain alert for warning signs that may require professional evaluation. While broad best practices are well supported, some research gaps remain in identifying the most effective break frequency and long-term outcomes of specific interventions.

---

### Sources

[1] 5 Best Desk-Worker Neck Relief Methods That Work | Hyperhealth: https://www.hyperhealth.com.au/post/5-best-desk-worker-neck-relief-methods-that-work  
[2] Reducing Neck and Back Pain at Work - Spine-health: https://www.spine-health.com/blog/reducing-neck-and-back-pain-work  
[3] Workplace-Based Interventions for Neck Pain in Office Workers: https://espace.library.uq.edu.au/view/UQ:693045  
[4] Neck and Spine Exercises for Desk Job Employees | BIOKINETIX: https://biokinetix.com/blog/2025/02/05/neck-and-spine-exercises-desk-job/  
[5] Reducing Sedentary Behavior to Decrease Chronic Low Back Pain: https://pmc.ncbi.nlm.nih.gov/articles/PMC8283944/  
[6] Top Mobility Exercises for Desk Workers - Delaware Fit Factory: https://www.delawarefitfactory.com/blog/top-mobility-exercises-for-desk-workers  
[7] Top 12 Mobility Exercises to Do at Your Desk | Sunny Health & Fitness: https://sunnyhealthfitness.com/blogs/health-wellness/mobility-desk-exercises?srsltid=AfmBOoonHpKW2tMU2ZBQfpQ249Bs26w9o0XNeISnNI5m9LOhz3iru5i5  
[8] The Ultimate 10-Minute Daily Mobility Routine for Beginners in 2025: https://vitaladjustment.com/the-ultimate-10-minute-daily-mobility-routine-for-beginners-in-2025  
[9] 6 Desk Exercises That Help You Get Stronger While Working - Cleveland Clinic: https://health.clevelandclinic.org/desk-exercises  
[10] Fitness for People Who Work at Desks: A Lifestyle-Based Approach: https://originsunity.com/fitness-for-people-who-work-at-desks-a-lifestyle-based-approach/  
[13] The Ultimate 10-Minute Daily Mobility Routine for Beginners in 2025: https://vitaladjustment.com/the-ultimate-10-minute-daily-mobility-routine-for-beginners-in-2025  
[16] Office Ergonomics Mistakes You Might Be Making (And How to Fix ...): https://ewiworks.com/office-ergonomics-mistakes-you-might-be-making-and-how-to-fix-them/  
[17] Ergonomic Tips for Desk Workers | Prevent Strain & Boost Productivity: https://www.opaortho.com/ergonomic-tips-for-desk-workers/  
[18] Deskworker ergonomics and taking breaks: https://www.brunswickosteopathy.com.au/blog/deskworker-ergonomics-and-taking-breaks  
[21] Tips for Office Workers to Avoid Back Pain: https://www.isppcenter.com/blog/tips-for-office-workers-to-avoid-back-pain  
[24] Back Pain From Sitting All Day: 10 Desk Job Tips to Protect Your Spine: https://www.michiganneurologyassociates.com/blog/back-pain-from-sitting-all-day-10-desk-job-tips-to-protect-your-spine  
[25] Addressing Neck and Back Pain When You're Working from Home: https://www.hopkinsmedicine.org/health/conditions-and-diseases/back-pain/addressing-neck-and-back-pain-when-youre-working-from-home  
[28] Work-break interventions for preventing musculoskeletal symptoms ... - Cochrane Review: https://www.cochrane.org/evidence/CD012886_work-break-interventions-preventing-musculoskeletal-symptoms-and-disorders-healthy-workers  
[29] Work‐break schedules for preventing musculoskeletal symptoms ... - Cochrane/Systematic Review: https://pmc.ncbi.nlm.nih.gov/articles/PMC6646952/  
[30] Ergonomic Interventions For Reducing Musculoskeletal Disorders In ... - IJFMR Systematic Review: https://www.ijfmr.com/papers/2025/6/62905.pdf


Research workflow completed!


#### Reflection on the wellness report generated:

**What went well?**
- The clarification step correctly identifed that my request was clear and it had sufficient information to proceed with research as seen from LangSmith.
- The final rpeort generated was well structured, with clear sections and actions that I can follow.

**What could be improved?**
- For production use, add safety guardrails and clear medical disclaimers.
- May be add SKILLS.md or give a healthWellnessGuide with personalized details for my age and preferences (fitness level, previous injuries etc).